# MedScope AI — Colab Inference and API

Run the backend API server on Google Colab using your trained weights from Google Drive.

In [ ]:
#@title 1. Runtime and configuration
import os, sys
REPO_URL = "https://github.com/prey801/finalyearproject.git"
BRANCH = "main"
PROJECT_DIR = "/content/finalyearproject"
USE_DRIVE = True
DRIVE_WEIGHTS_DIR = "/content/drive/MyDrive/medscope-ai/weights"
NGROK_AUTH_TOKEN = "" #@param {type:"string"}


In [ ]:
#@title 2. Mount Google Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
#@title 3. Clone repository
!git clone --branch "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
%cd $PROJECT_DIR

In [ ]:
#@title 4. Install dependencies
!apt-get update -qq
!apt-get install -y -qq redis-server postgresql libgl1
!python -m pip install -U pip setuptools wheel
!python -m pip install -r backend/requirements.txt
!python -m pip install -r models/requirements.txt
!python -m pip install pytest qdrant-client pyngrok nest_asyncio

In [ ]:
#@title 5. Load Trained Weights from Drive
import shutil
from pathlib import Path

if USE_DRIVE:
    src_dir = Path(DRIVE_WEIGHTS_DIR)
    dst_dir = Path(PROJECT_DIR) / "models/weights"
    dst_dir.mkdir(parents=True, exist_ok=True)
    for p in src_dir.glob("*.pt"):
        print(f"Copying {p.name}...")
        shutil.copy2(p, dst_dir / p.name)
    print("Weights loaded!")

In [ ]:
#@title 6. Start Infrastructure Services
!service redis-server start
!service postgresql start
!sudo -u postgres psql -c "CREATE USER medscope WITH PASSWORD 'password';"
!sudo -u postgres psql -c "CREATE DATABASE medscope_db OWNER medscope;"

!wget -q https://github.com/qdrant/qdrant/releases/download/v1.7.0/qdrant-x86_64-unknown-linux-gnu.tar.gz
!tar -xzf qdrant-x86_64-unknown-linux-gnu.tar.gz
import subprocess, time
subprocess.Popen(["./qdrant"])
time.sleep(3)

In [ ]:
#@title 7. Run Backend API via ngrok
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import os
from backend.main import app

if not NGROK_AUTH_TOKEN:
    raise ValueError("Please provide an NGROK_AUTH_TOKEN in step 1")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print(f"\n\n>>> API is available at: {public_url.public_url} <<<\n\n")

os.environ['YOLO_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "models/weights/yolov11_malaria_best.pt")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)